## [Locus: the paper](https://antfriend.github.io/locus_arxiv.pdf)
for this novel framework.   

The learnings:
## [Locus Logs](https://antfriend.github.io/index.html?ttdb=companion_arcprize.md&toot=lat-300lon10)

In [ ]:
# learn better
import os, subprocess

# Dump Kaggle/competition env vars for diagnostics
for k, v in sorted(os.environ.items()):
    if any(k.startswith(p) for p in ('KAGGLE', 'ARC', 'COMPETITION', 'GATEWAY')):
        print(f'[env] {k}={v}')

# Install competition wheels
_wdir = '/kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels'
if not os.path.isdir(_wdir):
    _wdir = '/kaggle/input/datasets/danxray/companion-arc/wheels'
print(f'[wheels] {_wdir}')
import glob as _g
subprocess.run(['pip', 'install', '--no-deps', '-q'] + _g.glob(f'{_wdir}/*.whl'), check=True)

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    # -------------------------------------------------------------------
    # COMPETITION RERUN — use official ARC-AGI-3-Agents framework
    # with LOCUS route agent
    # -------------------------------------------------------------------
    print('[mode] COMPETITION RERUN — framework mode (locus agent)')

    # Write LOCUS route agent
    _locus_agent = '''
import random as _random
from typing import Any
from arcengine import FrameData, GameAction, GameState
from agents.agent import Agent


class LucusAgent(Agent):
    """LOCUS-discovered routes for ARC-AGI-3 competition games.
    Phase 1: execute known route. Phase 2: random play until WIN.
    """

    MAX_ACTIONS = float('inf')

    _ROUTES = {
        "ls20": [0, 0, 0, 0, 2, 2, 2, 1, 0, 3, 3, 3, 0, 0, 0],
        "cd82": [3, 0, 1, 0, 0, 0, 1, 1, 1, 3, 2, 0, 4, 4, 2, 0, 0, 0, 1],
        "sp80": [4, 3, 3, 3, 4, 2, 2, 1],
    }

    def __init__(self, *args: Any, **kwargs: Any) -> None:
        super().__init__(*args, **kwargs)
        prefix = self.game_id.split("-")[0] if "-" in self.game_id else self.game_id
        self._route = self._ROUTES.get(prefix, [])
        self._play_step = 0
        self._simple = [a for a in GameAction if a.is_simple()]
        self._rng = _random.Random(hash(self.game_id) & 0xFFFFFFFF)
        print(f"[locus] {self.game_id}: route={len(self._route)} steps, actions={len(self._simple)}")

    def is_done(self, frames: list, latest_frame: FrameData) -> bool:
        return latest_frame.state is GameState.WIN

    def choose_action(self, frames: list, latest_frame: FrameData) -> GameAction:
        if latest_frame.state in (GameState.NOT_PLAYED, GameState.GAME_OVER):
            self._play_step = 0
            return GameAction.RESET

        if self._play_step < len(self._route):
            idx = self._route[self._play_step] % max(len(self._simple), 1)
            action = self._simple[idx]
            if action.is_simple():
                action.reasoning = f"LOCUS route step {self._play_step}"
            self._play_step += 1
            return action

        # Route exhausted — random play until WIN
        action = self._rng.choice(self._simple) if self._simple else GameAction.RESET
        if action.is_simple():
            action.reasoning = "LOCUS random fallback"
        return action
'''
    with open('/kaggle/working/locus_agent.py', 'w') as _f:
        _f.write(_locus_agent)

    # Wait for gateway to be ready
    subprocess.run([
        'curl', '--fail', '--retry', '999', '--retry-all-errors',
        '--retry-delay', '5', '--retry-max-time', '600',
        'http://gateway:8001/api/games'
    ], check=True)

    # Copy framework to writable location
    subprocess.run([
        'cp', '-r',
        '/kaggle/input/competitions/arc-prize-2026-arc-agi-3/ARC-AGI-3-Agents',
        '/kaggle/working/ARC-AGI-3-Agents'
    ], check=True)

    # Install locus agent into framework
    subprocess.run([
        'cp', '/kaggle/working/locus_agent.py',
        '/kaggle/working/ARC-AGI-3-Agents/agents/templates/locus_agent.py'
    ], check=True)

    # Override __init__.py — original has unmet deps that crash at import
    with open('/kaggle/working/ARC-AGI-3-Agents/agents/__init__.py', 'w') as _f:
        _f.write(
            'from typing import Type\n'
            'from dotenv import load_dotenv\n'
            'from .agent import Agent, Playback\n'
            'from .swarm import Swarm\n'
            'from .templates.random_agent import Random\n'
            'from .templates.locus_agent import LucusAgent\n'
            '\n'
            'load_dotenv()\n'
            '\n'
            'AVAILABLE_AGENTS: dict[str, Type[Agent]] = {\n'
            "    'random': Random,\n"
            "    'locus': LucusAgent,\n"
            '}\n'
        )

    # Write .env — gateway config
    with open('/kaggle/working/ARC-AGI-3-Agents/.env', 'w') as _f:
        _f.write(
            'SCHEME=http\n'
            'HOST=gateway\n'
            'PORT=8001\n'
            'ARC_API_KEY=locus_agent\n'
            'ARC_BASE_URL=http://gateway:8001/\n'
            'OPERATION_MODE=online\n'
            'ENVIRONMENTS_DIR=\n'
            'RECORDINGS_DIR=/kaggle/working/server_recording\n'
        )

    # Run LOCUS agent via framework
    _env = {**os.environ, 'MPLBACKEND': 'agg'}
    subprocess.run(
        ['python', 'main.py', '--agent', 'locus'],
        cwd='/kaggle/working/ARC-AGI-3-Agents',
        env=_env,
        check=False
    )

else:
    # -------------------------------------------------------------------
    # BATCH RUN — offline diagnostics + parquet (not used for scoring)
    # -------------------------------------------------------------------
    print('[mode] BATCH RUN — offline diagnostics')
    subprocess.run([
        'python3',
        '/kaggle/input/datasets/danxray/companion-arc/launch_competition.py'
    ], env=os.environ, check=False)
